# Track 2 — Intelligent Well Archive

Optional **local inspection and keyword-retrieval baseline**, not a finished question-answering system. PDF parsing, evidence-backed answers and a reproducible solution notebook are participant responsibilities. Use the approved dataset provided by the organisers. The notebook locates it when run from the repository root or this starter-kit directory; otherwise set `DATA_ROOT` to its location.

In [ ]:
from pathlib import Path
import csv
import re
from html import unescape

# Resolve data from either the repository root or the starter-kit directory.
def find_data_root():
    candidates = [
        Path.cwd() / "data" / "track_2",
        Path.cwd() / "../../data/track_2",
        Path.cwd() / "../data/track_2",
    ]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot locate data/track_2. Run from the repository root or the track starter-kit "
        "directory, or set DATA_ROOT to the approved dataset location."
    )

DATA_ROOT = find_data_root()
MANIFEST = DATA_ROOT / "metadata" / "document_manifest.csv"
DOCUMENTS = DATA_ROOT / "documents"
EXPECTED_COLUMNS = ["question_id", "answer", "evidence", "confidence"]
QUESTIONS = DATA_ROOT / "open_test" / "open_test_questions.csv"
with QUESTIONS.open(newline="", encoding="utf-8-sig") as handle:
    questions = list(csv.DictReader(handle))
print("Published questions:", len(questions))
print("Dataset found:", DATA_ROOT.exists())
print("Manifest found:", MANIFEST.exists())
print("Documents folder found:", DOCUMENTS.exists())

# Load the shared submission checker (submission/validate_submission.py in this repository).
import sys
import json

def find_validator_dir():
    for base in (Path.cwd(), Path.cwd() / "..", Path.cwd() / "../.."):
        folder = base / "submission"
        if (folder / "validate_submission.py").is_file():
            return folder.resolve()
    raise FileNotFoundError(
        "Cannot locate submission/validate_submission.py. Run from the repository root or the track starter-kit directory."
    )

sys.path.insert(0, str(find_validator_dir()))
import validate_submission as vs
print("Submission checker loaded:", vs.__file__)

In [ ]:
if MANIFEST.is_file():
    with MANIFEST.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        print("Manifest columns:", reader.fieldnames)
        print("Sample manifest rows:", [row for _, row in zip(range(3), reader)])
else:
    print("Manifest unavailable at placeholder path; confirm approved data root with technical support.")

In [ ]:
def read_local_text(path):
    if path.suffix.lower() not in {".txt", ".html", ".htm", ".xml"}:
        raise ValueError("This basic reader supports only text-like files; add a PDF parser separately.")
    raw = path.read_text(encoding="utf-8", errors="replace")
    if path.suffix.lower() in {".html", ".htm", ".xml"}:
        raw = unescape(re.sub(r"<[^>]+>", " ", raw))
    return raw

files = sorted(
    p for p in DOCUMENTS.rglob("*")
    if p.is_file() and p.suffix.lower() in {".txt", ".html", ".htm", ".xml"}
) if DOCUMENTS.is_dir() else []
print("Readable text-like files:", len(files))
if files:
    print("Preview from", files[0].relative_to(DATA_ROOT), ":", read_local_text(files[0])[:350])
print("PDF content is not covered by this baseline; use an approved local PDF parser.")

In [ ]:
def keyword_search(question, paths, top_k=3):
    stopwords = {"the", "a", "an", "in", "on", "of", "is", "what", "when", "which"}
    terms = set(re.findall(r"\w+", question.lower())) - stopwords
    matches = []
    for path in paths:
        content = read_local_text(path)
        score = sum(len(re.findall(r"\b" + re.escape(term) + r"\b", content.lower())) for term in terms)
        if score:
            matches.append((score, str(path.relative_to(DATA_ROOT)), content[:250]))
    return sorted(matches, key=lambda x: (-x[0], x[1]))[:top_k]

print(keyword_search("well drilling report", files))

## Validate your own answers CSV

The example search results are **not answers** and must not be treated as verified evidence. Use the 32 questions in `DATA_ROOT / "open_test/open_test_questions.csv"` and your own QA workflow to generate predictions. Copy the supplied `track2_predictions_template.csv`, keep all question IDs, and save it as your team's `submission/predictions.csv`. Judges accept equivalent correct wording and review source-file/page evidence under the Submission Guide rubric.

In [ ]:
PREDICTIONS = Path("submission/predictions.csv")  # set after implementing your solution
if PREDICTIONS.is_file():
    result = vs.validate_csv(PREDICTIONS, 2, vs.load_expected(2, DATA_ROOT))
    print(json.dumps(result, indent=2))
    with PREDICTIONS.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        rows = list(reader)
        print("Answers without citations (review abstentions separately):", sum(bool(row.get("answer", "").strip()) and not row.get("evidence", "").strip() for row in rows))
else:
    print("Validator skipped: generate your own answers CSV first.")

## Your implementation

Add approved PDF parsing, retrieval or extraction, grounded answering, evidence paths, abstention and confidence. Save any fitted indexes/models before submission. Ensure your submitted notebook can reproduce its outputs using the approved archive and declared dependencies.

## Validate Your Submission (optional)

Set `SUBMISSION_DIR` to your private team repository's `submission/` folder, then run the cell below. It checks required files, columns and identifiers, not answer quality or solution execution. The same check runs from a terminal at the repository root:

```bash
python submission/validate_submission.py --track 2 ../nape-aice-2026-pipeline-x/submission
```

In [ ]:
# Example only: replace pipeline-x and adjust for your current working directory.
SUBMISSION_DIR = Path("../nape-aice-2026-pipeline-x/submission")
report = vs.validate_submission(SUBMISSION_DIR, track=2, data_root=DATA_ROOT)
print(json.dumps(report, indent=2))
print("PASS: format valid" if report["valid"] else "FAIL: fix errors above")